# QAT · Text-only GRPO · Arithmetic

Try `mlx-community/Qwen3.5-2B-4bit` with weight-only affine QAT through the
shared training API. The loader reads the packed checkpoint; preparation
**dequantizes it to floating-point master weights**, then simulates 4-bit
quantization on eligible standard Linear layers with straight-through gradients.
Embeddings and unsupported layers stay floating point. This does not recover
the original pre-quantization weights.

This is full fine-tuning, including the vision tower, with full-training memory
and optimizer costs. Text-only batches provide no vision inputs. Start with
these four-step settings, then increase sample limits and iterations.
Checkpoints contain float master weights, not packed 4-bit inference weights.
Resume with the same QAT settings. The final export cell saves a packed inference model.

Arithmetic GRPO uses `mlx-community/simple-grpo` with correctness and reasoning-tag
format rewards. Answers stay outside the prompts. `beta=0` avoids a second model;
set a positive beta for a frozen quantized reference. W&B is disabled by default.

## Setup

Use an Apple Silicon Mac. Install into the notebook kernel, then restart it:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel pandas
```

Run cells in order. Model and dataset downloads are cached after first use.

Use the notebook kernel's `%pip install -e "/absolute/path/to/mlx-vlm[train]"`
if an installed copy lacks `export_model`, then restart the kernel **before**
training. The first import cell checks that this API is available.


In [ ]:
import json
import re
from dataclasses import asdict, replace
from decimal import Decimal, InvalidOperation
from pathlib import Path

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset

from mlx_vlm import export_model, load
from mlx_vlm.trainer import (
    GRPOTrainingArgs,
    Trainer,
    WandBCallback,
    prepare_model_for_training,
)
from mlx_vlm.trainer.vlm.grpo.trainer import generate_grpo


## 1. Settings

Four microsteps provide a short smoke run. Increase iters and sample limits for
longer experiments. QAT uses full fine-tuning; keep TRAIN_TYPE="full" and QAT=True.
QAT_BITS accepts 4 or 8, and QAT_GROUP_SIZE accepts 32, 64, or 128. GRPO's group_size
counts sampled completions and is separate from the QAT quantization group size.
Paths are relative to the notebook kernel's working directory.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/simple-grpo"
OUTPUT_DIR = Path("checkpoints/vlm/qat_grpo_text_only")
WANDB_PROJECT = None  # Optional: set a project name to log to W&B.
MAX_TRAIN_SAMPLES = 32
MAX_EVAL_SAMPLES = 4
MAX_TEST_SAMPLES = 4
EXPORT_DIR = OUTPUT_DIR / "quantized"  # Must be a new directory.
SEED = 42
TRAIN_TYPE = "full"
QAT = True
QAT_BITS = 4
QAT_GROUP_SIZE = 64
CHECKPOINT_PATH = None

FORMAT_INSTRUCTION = (
    "A conversation between User and Assistant. The user asks a question, and the "
    "Assistant solves it. The assistant first thinks about the reasoning process in "
    "the mind and then provides the user with the answer. The reasoning process and "
    "answer are enclosed within <think> </think> and <answer> </answer> tags, "
    "respectively, i.e., <think> reasoning process here </think> <answer> answer here "
    "</answer>."
)
# GRPO variants share this trainer; choose their loss normalization below:
#   "grpo": mean over tokens per completion, then mean over completions.
#   "bnpo": divide the summed loss by total completion tokens.
#   "dr_grpo": divide by completion count * max_completion_length.
# Rewards always use group standard-deviation scaling, including "dr_grpo";
# this option selects the denominator, not every change in the Dr. GRPO method.
# "sequence" importance sampling gives GSPO-style ratios; epsilon_high
# sets a DAPO-style upper clipping width (defaults to epsilon when None).
# Each rollout is used once: ratios start at 1 and clipping normally stays 0.
# No multi-epoch PPO or DAPO dynamic resampling is implemented.
# beta controls reference-model KL; beta=0 skips reference scoring.
# advantage_epsilon stabilizes reward scaling; min_p also controls sampling.
R1_PROMPT_SOURCE = "https://arxiv.org/html/2501.12948v1#S2.T1"

args = GRPOTrainingArgs(
    iters=4,
    batch_size=1,
    group_size=2,
    max_completion_length=128,
    max_seq_length=1024,
    temperature=0.5,
    top_p=0.95,
    top_k=20,
    beta=0.0,
    epsilon=0.2,
    grpo_loss_type="grpo",  # also "bnpo" or "dr_grpo"
    importance_sampling_level="token",  # or "sequence" (GSPO-style)
    end_answer_token="</answer>",
    gradient_accumulation_steps=1,
    grad_checkpoint=True,
    logit_chunk_size=32,
    cache_size=32,
    compile=False,
    val_batches=1,
    steps_per_report=1,
    steps_per_eval=4,
    steps_per_save=4,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Keep the supplied data splits separate

simple-grpo supplies train (400 rows), valid (50), and test (50).
Training uses only train; progress checks use valid; the final cell uses test.
Targets are retained only as reward metadata. Mapping does not inject the
answer or an example solution into the prompt.


In [ ]:
raw_dataset = load_dataset(DATASET_ID)
required_splits = {"train", "valid", "test"}
if not required_splits.issubset(raw_dataset):
    raise ValueError(f"Expected train/valid/test splits, got {list(raw_dataset)}.")


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


def mapp_func(sample):
    question, answer = sample.get("prompt"), sample.get("answer")
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Each question must contain a non-empty prompt string.")
    if answer is None or not str(answer).strip():
        raise ValueError("Every arithmetic question needs a numeric answer.")
    try:
        number = Decimal(str(answer).strip())
    except InvalidOperation as error:
        raise ValueError("Each answer must be a finite number.") from error
    if not number.is_finite():
        raise ValueError("Each answer must be a finite number.")
    prompt = f"{FORMAT_INSTRUCTION}\nUser: {question}\nAssistant:"
    return {"prompt": prompt, "question": question, "answer": str(answer).strip()}


def prepare_rows(rows, maximum):
    rows = limit_rows(rows, maximum)
    return rows.map(mapp_func, remove_columns=rows.column_names)


train_rows = prepare_rows(raw_dataset["train"], MAX_TRAIN_SAMPLES)
eval_rows = prepare_rows(raw_dataset["valid"], MAX_EVAL_SAMPLES)
test_rows = prepare_rows(raw_dataset["test"], MAX_TEST_SAMPLES)
assert min(len(train_rows), len(eval_rows), len(test_rows)) >= args.batch_size
print({"train": len(train_rows), "validation": len(eval_rows), "test": len(test_rows)})
print(train_rows[0])


## 3. Rule-based rewards

Accuracy is the main signal (weight 1.0). A smaller format reward (weight 0.1)
encourages one non-empty reasoning block followed by one numeric answer block.
The reward verifies the final answer, not the quality of the reasoning text.

The correctness checker accepts a single answer block or a bare numeric
response, so the policy can receive correctness feedback before it learns the
format. It rejects duplicate answer blocks, incomplete tags, and nonnumeric
answers; Decimal equality accepts equivalent forms such as 139 and 139.0.
This parser and the weights are choices for this tutorial.


In [ ]:
NUMBER_PATTERN = re.compile(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?")
ANSWER_PATTERN = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.DOTALL)
FORMAT_PATTERN = re.compile(
    r"\s*<think>(?P<reasoning>.*?)</think>\s*<answer>(?P<answer>.*?)</answer>\s*",
    re.DOTALL,
)
TAGS = ("<think>", "</think>", "<answer>", "</answer>")


def extract_answer(completion):
    blocks = ANSWER_PATTERN.findall(completion)
    if blocks:
        if (
            len(blocks) != 1
            or completion.count("<answer>") != 1
            or completion.count("</answer>") != 1
        ):
            return None
        text = blocks[0].strip()
    else:
        text = completion.strip()
    if NUMBER_PATTERN.fullmatch(text) is None:
        return None
    try:
        number = Decimal(text)
    except InvalidOperation:
        return None
    return number if number.is_finite() else None


def correctness_reward(*, completions, answer, **kwargs):
    scores = []
    for completion, target in zip(completions, answer):
        predicted = extract_answer(completion)
        scores.append(
            float(predicted is not None and predicted == Decimal(str(target).strip()))
        )
    return scores


def reasoning_format_reward(*, completions, **kwargs):
    scores = []
    for completion in completions:
        match = FORMAT_PATTERN.fullmatch(completion)
        valid = (
            match is not None
            and all(completion.count(tag) == 1 for tag in TAGS)
            and bool(match.group("reasoning").strip())
            and NUMBER_PATTERN.fullmatch(match.group("answer").strip()) is not None
        )
        scores.append(float(valid))
    return scores


reward_funcs = [correctness_reward, reasoning_format_reward]
reward_weights = [1.0, 0.1]
print(
    correctness_reward(
        completions=["<think>56 + 83 = 139.</think><answer>139</answer>"],
        answer=["139"],
    )
)
print(
    reasoning_format_reward(
        completions=["<think>56 + 83 = 139.</think><answer>139</answer>"]
    )
)


## Prepare full QAT

Load the requested packed model directly. Preparation dequantizes packed weights,
then replaces compatible standard Linear layers with local QAT layers. No global
class patches or separate training loop are needed. Set CHECKPOINT_PATH to a full
checkpoint to resume with a fresh optimizer and the same QAT settings.


In [ ]:
model, processor = load(MODEL_ID)
reference_model = load(MODEL_ID)[0] if args.beta > 0 else None
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    qat=QAT,
    quantization_bits=QAT_BITS,
    quantization_group_size=QAT_GROUP_SIZE,
    checkpoint_path=CHECKPOINT_PATH,
)


### Use the mapped prompt directly

mapp_func builds the published DeepSeek instruction, the question, and the
Assistant: suffix in the prompt column. The trainer tokenizes that string directly
with dataset_config={"apply_chat_template": False}; no chat-template override is
needed. The answer stays in its own column for rewards and never enters the
model prompt. The question column preserves the original text for inspection.


In [ ]:
training_tokenizer = getattr(processor, "tokenizer", processor)

optimizer = optim.AdamW(
    learning_rate=1e-5,
    weight_decay=0.0,
)

trainer = Trainer(
    model,
    args,
    optimizer=optimizer,
    algorithm="grpo",
    processing_class=processor,
    dataset_config={"apply_chat_template": False},
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    reference_model=reference_model,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
    seed=SEED,
)
prepared_sample = trainer.train_dataset[0]
print(training_tokenizer.decode(prepared_sample["inputs"]["input_ids"].tolist()))
assert "pixel_values" not in prepared_sample["inputs"]


## 5. Baseline and training

Validation samples fresh groups. Compare correctness and format reward means,
not only GRPO loss. Groups with identical rewards have zero policy advantage;
monitor grouped_reward_std. If the policy receives almost no reward,
inspect its responses and adjust sampling, token budget, or question difficulty.

The memory optimizations match the vision GRPO path: CPU reward normalization,
completion-only vocabulary projection, synchronized gradients per completion,
and transformer checkpointing. With beta > 0, KL uses the frozen starting checkpoint.


In [ ]:
baseline = trainer.evaluate()
print(
    {
        key: value
        for key, value in baseline.items()
        if "reward" in key or key == "val_loss"
    }
)
wandb_callback = None
if WANDB_PROJECT:
    wandb_callback = WandBCallback(
        project_name=WANDB_PROJECT,
        log_dir=str(OUTPUT_DIR),
        config={
            "model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "seed": SEED,
            "train_type": TRAIN_TYPE,
            "qat": QAT,
            "quantization_bits": QAT_BITS,
            "quantization_group_size": QAT_GROUP_SIZE,
            "reward_weights": reward_weights,
            **asdict(args),
        },
    )
    trainer.training_callback = wandb_callback

try:
    metrics = trainer.train()
finally:
    if wandb_callback is not None:
        wandb_callback.finish()
assert mx.isfinite(mx.array(metrics["train_loss"])).item()
reports = pd.DataFrame(trainer.log_history)
reports.tail(5).reindex(
    columns=[
        "iteration",
        "train_loss",
        "train_reward_0_correctness_reward",
        "train_reward_1_reasoning_format_reward",
        "train_grouped_reward_std",
        "train_kl",
        "trained_tokens",
        "peak_memory",
    ]
)


## 6. Compare validation and inspect a generated group

Correctness reward is the sampled numeric accuracy; reasoning format reward
is the fraction of responses matching the requested tag structure. Use more
validation batches for a steadier estimate. These figures measure this small
addition dataset, not general reasoning performance.


In [ ]:
validation = trainer.evaluate()
comparison = pd.DataFrame([baseline, validation], index=["before GRPO", "after GRPO"])
comparison.reindex(
    columns=[
        "val_reward_0_correctness_reward",
        "val_reward_1_reasoning_format_reward",
        "val_reward_mean",
        "val_grouped_reward_std",
        "val_kl",
    ]
).round(4)


In [ ]:
sample = trainer.eval_dataset[0]
completion_ids, completion_texts, _ = generate_grpo(
    model,
    training_tokenizer,
    [sample],
    args,
)
print("Question:", sample["metadata"]["question"])
print("Target:", sample["answer"])
for index, text in enumerate(completion_texts):
    accuracy = correctness_reward(completions=[text], answer=[sample["answer"]])[0]
    format_score = reasoning_format_reward(completions=[text])[0]
    print(f"Sample {index + 1} (correct={accuracy}, format={format_score}):\n{text}")


## 7. Final held-out test

Use the supplied test partition only after training/settings decisions.
This cell evaluates all selected test rows rather than the short validation budget.


In [ ]:
test_args = replace(args, val_batches=-1)
test_trainer = Trainer(
    model,
    test_args,
    algorithm="grpo",
    processing_class=processor,
    dataset_config={"apply_chat_template": False},
    eval_dataset=test_rows,
    reference_model=reference_model,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
)
test_metrics = test_trainer.evaluate()
pd.DataFrame([test_metrics]).reindex(
    columns=[
        "val_reward_0_correctness_reward",
        "val_reward_1_reasoning_format_reward",
        "val_reward_mean",
        "val_kl",
    ]
).round(4)


## 8. Save full master weights and prompt settings

The loop saves a full checkpoint automatically. Keep weights.safetensors and
adapter_config.json together, plus the prompt template and run_config.json.
The run config records QAT settings; resume with the same settings and a fresh optimizer.


In [ ]:
trainer.save_adapter()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prompt_template = f"{FORMAT_INSTRUCTION}\nUser: {{question}}\nAssistant:"
(OUTPUT_DIR / "prompt_template.txt").write_text(prompt_template, encoding="utf-8")
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "base_model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "format_instruction": FORMAT_INSTRUCTION,
            "prompt_template": prompt_template,
            "dataset_config": {"apply_chat_template": False},
            "prompt_source": R1_PROMPT_SOURCE,
            "train_type": TRAIN_TYPE,
            "qat": QAT,
            "quantization_bits": QAT_BITS,
            "quantization_group_size": QAT_GROUP_SIZE,
            "reward_weights": reward_weights,
            "training_args": asdict(args),
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print(Path(args.adapter_file).resolve())


### Reload with the same QAT settings

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type="full", qat=True,
    quantization_bits=QAT_BITS, quantization_group_size=QAT_GROUP_SIZE,
    checkpoint_path=args.adapter_file,
)
model.eval()
```

Evaluation retains fake quantization. Full checkpoints record QAT settings, which model preparation restores automatically. For GRPO reuse
the saved prompt_template.txt and dataset_config={"apply_chat_template": False}.
See the [training guide](../../docs/training.md).


## Export packed QAT weights for inference

`export_model` automatically detects QAT layers and writes a standalone model using the exact bit/group settings of each QAT
layer, plus the base config and processor. Other layers keep their training
representation. Float checkpoints above remain available for resuming training.
Use a new EXPORT_DIR for each export. The live model keeps its float master weights.

Load the exported directory directly with `load(str(EXPORT_DIR))`; do not attach
it as an adapter or run model preparation for inference.

Use `quantize=True` to also quantize compatible non-QAT layers (defaults: 4 bits, group size 64).


In [ ]:
from mlx_vlm import export_model

export_path = export_model(
    model, processor, EXPORT_DIR, model_path=MODEL_ID,
)
print("Packed inference model:", export_path.resolve())
# Later, in a fresh kernel:
# inference_model, inference_processor = load(str(export_path))
